# Train the bird -> species models

1. **Runtime > Change runtime type > T4 GPU** (do this first).
2. Run the cells in order. When the upload cell appears, choose all three files:
   `prepare_data.py`, the herons zip and the ducks zip (any file names are fine as long as they contain 'heron' and 'duck').
3. At the end, `bird_det.pt` and `species.pt` download to your computer.

In [ ]:
!pip install -q ultralytics

In [ ]:
from google.colab import files
uploaded = files.upload()   # prepare_data.py + the two dataset zips

## Build the training data

In [ ]:
import glob
zips = glob.glob('*.zip')
herons = next((z for z in zips if 'heron' in z.lower()), None)
ducks = next((z for z in zips if 'duck' in z.lower()), None)
assert herons and ducks, f'Need one zip with heron in its name and one with duck. Found: {zips}'
print('herons:', herons, '| ducks:', ducks)
!python prepare_data.py --herons "{herons}" --ducks "{ducks}"


## Stage 1: bird detector (a few minutes to tens of minutes)

In [ ]:
import shutil
from ultralytics import YOLO

det = YOLO("yolo26n.pt")
det.train(data="prepared/bird_det/data.yaml", epochs=100, imgsz=640, scale=0.9,
          project="/content/runs", name="bird_det", exist_ok=True)
shutil.copy(det.trainer.best, "bird_det.pt")
print("saved bird_det.pt")

## Stage 2: species classifier

In [ ]:
cls = YOLO("yolo26n-cls.pt")
cls.train(data="prepared/species_cls", epochs=40, imgsz=224,
          project="/content/runs", name="species", exist_ok=True)
shutil.copy(cls.trainer.best, "species.pt")
print("saved species.pt")

## Download the two weight files

In [ ]:
files.download("bird_det.pt")
files.download("species.pt")